<a href="https://colab.research.google.com/github/Addychauhan/Weather-Temperature-Prediction/blob/main/weather_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Importing the Required Libararies

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

#Loading the Dataset

In [ ]:
df=pd.read_csv('/content/weatherHistory.csv')

In [ ]:
df

#Basic Information

In [ ]:
#Shape of the Dataset
df.shape

In [ ]:
#First 5 Records
df.head()

In [ ]:
#Last 5 Records
df.tail()

In [ ]:
#Data Types
df.info()

In [ ]:
#Statistical Summary
df.describe()

In [ ]:
#Number of Missing Values
df.isnull().sum()

In [ ]:
#Number of Duplicate Values
df.duplicated().sum()

#EDA

In [ ]:
import seaborn as sns
sns.set_style('whitegrid')

In [ ]:
plt.figure(figsize=(10,8))
sns.histplot(df['Temperature (C)'], bins=50,kde=True)
plt.title("Distribution of Temperature (C)")
plt.xlabel('Temperature (C)')
plt.ylabel("Count")
plt.tight_layout()
plt.show()

Temperature is roughly bell-shaped, ranging from about -22°C to 40°C, centered near 12°C.

In [ ]:
# num_cols=df.select_dtypes(include=np.number)
# num_cols

In [ ]:
num_cols = ['Temperature (C)','Apparent Temperature (C)','Humidity','Wind Speed (km/h)',
            'Wind Bearing (degrees)','Visibility (km)','Pressure (millibars)']

In [ ]:
plt.figure(figsize=(10,8))
corr=df[num_cols].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Correlation Heatmap of Numeric Features')
plt.tight_layout()
plt.show()

In [ ]:
df["Formatted Date"]=pd.to_datetime(df["Formatted Date"], utc=True)
# df["Month"]=df["Formatted Date"].dt.month
monthly=df.groupby(df["Formatted Date"].dt.month)["Temperature (C)"].mean()

plt.figure(figsize=(8,6))
sns.lineplot(x=monthly.index, y=monthly.values, marker='o', color="Blue")
plt.xlabel("Month")
plt.ylabel("Average Temperature (C)")
plt.tight_layout()
plt.show()

In [ ]:
sample=df.sample(3000, random_state=42)

plt.figure(figsize=(8,6))
sns.scatterplot(data=sample,
                x="Humidity", y="Temperature (C)", hue="Precip Type")
plt.title("Temperature Vs Humidity (Sampled)")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8,6))
sns.boxplot(data=df, x="Precip Type", y="Temperature (C)")
plt.title("Temperature by Precipitation Type")
plt.tight_layout()
plt.show()

As expected, `snow` days are notably colder than `rain` days.

#Data Cleaning and Feature Engineering

Steps:
- Drop rows with missing `Precip Type` (517 rows)
- Drop `Loud Cover` (constant, zero variance)
- Parse `Formatted Date` into `Month` and `Hour` (captures seasonality & daily cycle)
- Encode `Precip Type` as a binary flag `Precip_rain` (1 = rain, 0 = snow)
- Drop `Apparent Temperature (C)` from the feature set (data leakage, see EDA)
- Drop free-text columns `Summary` / `Daily Summary` (high-cardinality text, not used for this numeric regression task)

## Drop Missing Values present

Drops the 517 rows where Precip Type is missing (NaN). subset=['Precip Type'] means it only checks that one column — rows with missing values elsewhere would be kept. .copy() makes a fresh copy of the DataFrame so pandas doesn't warn about modifying a "view" of the original in the next steps.

In [ ]:
df = df.dropna(subset=['Precip Type']).copy()
df

##Remove column Loud Cover
Removes the Loud Cover column entirely, since it was found to be constant (always 0) during EDA — a column with zero variance gives a model nothing to learn from.

In [ ]:
df = df.drop(columns=['Loud Cover'])

##Fomat Date
Pulls just the hour (0–23) out of the datetime column. .dt is the accessor that unlocks date/time properties (like .dt.month, .dt.year) on a column that's already been converted to datetime64 type. This gives the model a way to learn daily patterns (e.g., cooler at 4am, warmer at 2pm).

In [ ]:
df = df.sort_values('Formatted Date').reset_index(drop=True)
df['Month'] = df['Formatted Date'].dt.month
df['Hour'] = df['Formatted Date'].dt.hour

In [ ]:
df['Precip_rain'] = (df['Precip Type'] == 'rain').astype(int)
df

In [ ]:
print("Shape after cleaning:", df.shape)

In [ ]:
#Drop Columns which don't play important role
df = df.drop(columns=['Apparent Temperature (C)'])

In [ ]:
df

In [ ]:
#Checking Missing values
df.isna().sum()

In [ ]:
df['Temp_lag1'] = df['Temperature (C)'].shift(1)
df['Temp_lag3'] = df['Temperature (C)'].shift(3)
df['Humidity_lag1'] = df['Humidity'].shift(1)
df['Temp_roll3'] = df['Temperature (C)'].shift(1).rolling(3).mean()
df = df.dropna().reset_index(drop=True)

In [ ]:
df

#Define Feature and Target Variable

In [ ]:
features = ['Humidity', 'Wind Speed (km/h)', 'Wind Bearing (degrees)',
            'Visibility (km)', 'Pressure (millibars)', 'Precip_rain', 'Month', 'Hour',
            'Temp_lag1', 'Temp_lag3', 'Humidity_lag1', 'Temp_roll3']
target = 'Temperature (C)'

In [ ]:
X = df[features].values
y = df[target].values
print("Feature matrix:", X.shape, " Target:", y.shape)

#Train-Test Split & Scaling

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler

In [ ]:
split = int(len(df) * 0.8)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

In [ ]:
# X_train, X_test, y_train, y_test=train_test_split(X,y, test_size=0.2, random_state=42)

scaler=StandardScaler()
X_train_s=scaler.fit_transform(X_train)
X_test_s=scaler.transform(X_test)

In [ ]:
X_train.shape[0]

In [ ]:
X_test.shape[0]

#Model Building

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [ ]:
results = {}

def evaluate(name, y_true, y_pred):
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    results[name] = {'RMSE': rmse, 'MAE': mae, 'R2': r2}
    print(f"{name:20s}  RMSE={rmse:.3f}  MAE={mae:.3f}  R2={r2:.4f}")

## Model 1: Linear Regression

In [ ]:
lr=LinearRegression()
lr

In [ ]:
lr.fit(X_train_s, y_train)
evaluate('Linear Regression', y_test, lr.predict(X_test_s))

##Model 2: Decision Tree

In [ ]:
dt = DecisionTreeRegressor(max_depth=10, random_state=42)
dt.fit(X_train, y_train)
evaluate('Decision Tree', y_test, dt.predict(X_test))

##Model 3: Random forest

In [ ]:
rf = RandomForestRegressor(n_estimators=150, max_depth=12, min_samples_leaf=3, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
evaluate('Random Forest', y_test, rf.predict(X_test))

##Model 4: Gradient Boost

In [ ]:
gb = GradientBoostingRegressor(n_estimators=100, max_depth=3, learning_rate=0.1, random_state=42)
gb.fit(X_train, y_train)
evaluate('Gradient Boosting', y_test, gb.predict(X_test))

##Model 5: Deep Learning (Neural Network)

In [ ]:
mlp = MLPRegressor(hidden_layer_sizes=(64, 32, 16), activation='relu', solver='adam',
                    max_iter=200, early_stopping=True, random_state=42)
mlp.fit(X_train_s, y_train)
evaluate('Deep Learning (MLP)', y_test, mlp.predict(X_test_s))

## Overfitting Check

Comparing Random Forest's score on data it trained on (`train`) vs. data it never saw (`test`).
If `train` were much higher than `test`, that would mean the model memorized rather than learned.

In [ ]:
print('RF - train:'); evaluate('RF train', y_train, rf.predict(X_train))
print('RF - test :'); evaluate('RF test', y_test, rf.predict(X_test))

Train R² (0.9941) and test R² (0.9932) are very close — **no meaningful overfitting**. That's a genuinely good sign, not an artifact.

## 5-Fold Cross-Validation

A single train/test split can be lucky. Cross-validation retrains the model 5 times on different slices of the
data and reports the spread of scores — this tells us how *stable* the result is.

In [ ]:
cv_scores = cross_val_score(
    RandomForestRegressor(n_estimators=80, max_depth=12, min_samples_leaf=3, random_state=42, n_jobs=-1),
    X, y, cv=5, scoring='r2'
)
print("CV R2 scores:", np.round(cv_scores, 4))
print("Mean:", round(cv_scores.mean(), 4), " Std:", round(cv_scores.std(), 4))

Very low standard deviation (0.0016) across folds — the ~0.99 R² is **stable**, not a fluke of one split.

#Model Comparison

In [ ]:
results_df = pd.DataFrame(results)
results_df.T.sort_values('RMSE')

In [ ]:
plt.figure(figsize=(8, 5))
results_df.T['RMSE'].sort_values().plot(kind='barh', color='#55A868')
plt.title('Model Comparison — RMSE with lag features (lower is better)')
plt.xlabel('RMSE (°C)')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
results_df.T['R2'].sort_values().plot(kind='barh', color='#4C72B0')
plt.title('Model Comparison — R² with lag features (higher is better)')
plt.xlabel('R2')
plt.tight_layout()
plt.show()

**Deep Learning (MLP) is now the best model** (RMSE ≈ 0.64°C, R² ≈ 0.995), just ahead of Linear Regression and Gradient Boosting. Notice Linear Regression jumped from R²=0.62 (v1) to R²=0.994 here — once `Temp_lag1` is in the feature set, even a simple straight-line model can 'ride along' the strong hour-to-hour trend.

#Feature Importance and Prediction Quality

In [ ]:
from sklearn.inspection import permutation_importance

result = permutation_importance(mlp, X_test_s, y_test, n_repeats=10, random_state=42, n_jobs=-1)
importances = pd.Series(result.importances_mean, index=features).sort_values()

plt.figure(figsize=(7,5))
importances.plot(kind='barh', color='#C44E52')
plt.title('Permutation Importance — Deep Learning (MLP)')
plt.tight_layout()
plt.show()

`Temp_lag1` dominates all other features by a wide margin for the Deep Learning model too — confirming recency is doing most of the work.

In [ ]:
preds = mlp.predict(X_test_s)   # was: rf.predict(X_test)

idx = np.random.choice(len(y_test), 2000, replace=False)
plt.figure(figsize=(5.5,5.5))
plt.scatter(y_test[idx], preds[idx], alpha=0.3, s=12, color='#4C72B0')
lims = [y_test.min(), y_test.max()]
plt.plot(lims, lims, 'r--', lw=1.5)
plt.xlabel('Actual'); plt.ylabel('Predicted')
plt.title('Deep Learning (MLP): Predicted vs Actual')
plt.tight_layout(); plt.show()

#Saving the best Model

In [ ]:
import joblib

#Compare all models by test RMSE, pick the lowest (Best)
fit_df = results_df.T.sort_values('RMSE')
best_name = fit_df['RMSE'].idxmin()
print("Best model:", best_name)

In [ ]:
best_model_map = {
    'Linear Regression': (lr, True),
    'Decision Tree': (dt, False),
    'Random Forest': (rf, False),
    'Deep Learning (MLP)': (mlp, True),
}

eligible_models_for_saving = [name for name in best_model_map.keys() if name in results_df.columns]
best_name = results_df.loc['RMSE', eligible_models_for_saving].idxmin()

best_model, needs_scaling = best_model_map[best_name]

In [ ]:
# 3. Save the model TOGETHER with the scaler and feature list
#    You need all three to make correct predictions later --
#    saving just the model alone is not enough.
joblib.dump({
    'model': best_model,
    'scaler': scaler,
    'features': features,
    'needs_scaling': needs_scaling,
    'model_name': best_name
}, 'best_weather_model.joblib')

print(f"Saved '{best_name}' to best_weather_model.joblib")